<a href="https://colab.research.google.com/github/subbareddypullalacheruvu/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-/blob/main/notebooks/feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sparkov Feature Engineering — Uplift Analysis

This notebook builds a naive baseline classifier on the Sparkov dataset, then
adds four engineered feature groups one at a time — category risk, hour-of-day,
amount, and implied velocity — recording the change in PR-AUC, F2, and MCC at
each step. Two encoding choices (hour, and the amount/velocity transforms)
were tested directly rather than assumed, since the shape of a feature's
relationship to fraud determines which encoding works.

## Setup

In [1]:
# 1. Kaggle download
import os
from getpass import getpass

os.environ['KAGGLE_USERNAME'] = input('Kaggle username: ')
os.environ['KAGGLE_KEY'] = getpass('Kaggle API key: ')

!kaggle datasets download -d mlg-ulb/creditcardfraud -p /content/data/ulb --unzip
!kaggle datasets download -d kartik2112/fraud-detection -p /content/data/sparkov --unzip

Kaggle username: Subbareddy_Subbz
Kaggle API key: ··········
Dataset URL: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
License(s): DbCL-1.0
100% 66.0M/66.0M [00:00<00:00, 81.1MB/s]

Dataset URL: https://www.kaggle.com/datasets/kartik2112/fraud-detection
License(s): CC0-1.0
100% 202M/202M [00:01<00:00, 118MB/s]



In [2]:

# 2. Clone the repo (already has split.py and utils.py inside it)
github_token = getpass('Paste your GitHub token: ')
repo_url = f'https://{github_token}@github.com/subbareddypullalacheruvu/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-.git'
!git clone {repo_url}

Paste your GitHub token: ··········
Cloning into 'Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-'...
remote: Enumerating objects: 86, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (78/78), done.
remote: Total 86 (delta 47), reused 18 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (86/86), 368.57 KiB | 6.83 MiB/s, done.
Resolving deltas: 100% (47/47), done.


In [3]:

# 3. Move in, confirm, move data in
%cd Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-
!mv /content/data/ulb /content/data/sparkov data/


/content/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-


In [4]:
import sys
sys.path.append('/content/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-')
from src.split import split_sparkov, split_ulb

sparkov_train, sparkov_test = split_sparkov()
ulb_train, ulb_test = split_ulb()


Train: 1,296,675 rows, from 2019-01-01 00:00:18 to 2020-06-21 12:13:37
Test:  555,719 rows, from 2020-06-21 12:14:25 to 2020-12-31 23:59:34
Train: 226,980 rows, from 0.0 to 145233.0
Test:  56,746 rows, from 145234.0 to 172792.0


In [5]:

print("Already in namespace:", [v for v in ["sparkov_train", "sparkov_test", "ulb_train", "ulb_test"] if v in dir()])

from src.split import split_sparkov
sparkov_train, sparkov_test = split_sparkov()

print(sparkov_train.shape, sparkov_test.shape)
print(sparkov_train.columns.tolist())
sparkov_train.head(2)

Already in namespace: []
Train: 1,296,675 rows, from 2019-01-01 00:00:18 to 2020-06-21 12:13:37
Test:  555,719 rows, from 2020-06-21 12:14:25 to 2020-12-31 23:59:34
(1296675, 23) (555719, 23)
['Unnamed: 0', 'trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud']


,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,...,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,...,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0


## Naive Baseline

In [6]:
# clean up + naive Sparkov baseline (raw numeric features only)

# 1. Drop the leftover index column - it's not a real feature
sparkov_train = sparkov_train.drop(columns=["Unnamed: 0"])
sparkov_test = sparkov_test.drop(columns=["Unnamed: 0"])

# 2. Sanity check - fraud rate should match the EDA numbers
print("Train fraud rate:", sparkov_train["is_fraud"].mean())
print("Test fraud rate:", sparkov_test["is_fraud"].mean())

# 3. Pick the raw, un-engineered features for the baseline
features = ["amt", "city_pop"]
target = "is_fraud"

X_train = sparkov_train[features]
y_train = sparkov_train[target]
X_test = sparkov_test[features]
y_test = sparkov_test[target]

# 4. Scale the features - Logistic Regression works better on similar-sized numbers
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # learn the scaling from train only
X_test_scaled = scaler.transform(X_test)         # apply that same scaling to test

# 5. Train a plain Logistic Regression - default settings, no class weighting
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

# 6. Score it with the project's three metrics (not accuracy - see EDA notes on why)
from sklearn.metrics import average_precision_score, fbeta_score, matthews_corrcoef

y_prob = model.predict_proba(X_test_scaled)[:, 1]   # predicted fraud probability
y_pred = (y_prob >= 0.5).astype(int)                # turn into 0/1 at the default threshold

print("PR-AUC:", average_precision_score(y_test, y_prob))
print("F2:", fbeta_score(y_test, y_pred, beta=2))
print("MCC:", matthews_corrcoef(y_test, y_pred))

Train fraud rate: 0.005788651743883394
Test fraud rate: 0.0038598644278853163
PR-AUC: 0.13658130213813538
F2: 0.0
MCC: -0.0015826530982826614


A Logistic Regression model was trained using only two raw columns, transaction amount and city population, with no feature engineering, no class rebalancing, and the default 0.5 decision threshold. This produced a naive baseline: PR-AUC 0.137, F2 0.0, MCC -0.0016. The test set's fraud rate (0.386%) was lower than the training set's (0.579%), an early sign of how imbalanced this problem is before any modelling begins. A PR-AUC of 0.137 is far below the ULB baseline's 0.73, but this is expected: ULB's raw features already carry more signal, while here the strongest raw signal (transaction category) was deliberately left out so it could be introduced later as an engineered feature. Even so, 0.137 is meaningfully above the 0.386% base rate, so the two weak features are not contributing nothing. F2 and MCC both landing near zero show that, at the default threshold, the model fails to catch almost any fraud. This row is treated as the fixed comparison point that every later feature group is measured against.

In [7]:
print("Predicted positives:", y_pred.sum(), "out of", len(y_pred))

Predicted positives: 359 out of 555719


To understand why F2 came out at exactly 0.0, the number of transactions the model flagged as fraud was checked: 359 out of 555,719. The test set contains roughly 2,145 actual fraud cases (0.386% of 555,719), so an F2 of 0.0 means none of the 359 flagged transactions were real frauds — every genuine fraud case was missed, and 359 false alarms were raised instead. This is expected for a model built on two features with no direct link to fraud behaviour, scored at an untuned threshold. It also previews the two directions the rest of this analysis addresses: better features, and imbalance-aware training so real fraud cases can actually surface.

## Category Risk

In [8]:
# add category risk as a feature, then re-run the same baseline steps

from src.utils import fit_category_risk, apply_category_risk

# 1. Learn the fraud rate per category from TRAIN ONLY
category_risk_lookup = fit_category_risk(sparkov_train, category_column="category", label_column="is_fraud")

# 2. Apply that same lookup to both train and test (test never influences the numbers)
sparkov_train["category_risk"] = apply_category_risk(sparkov_train, "category", category_risk_lookup)
sparkov_test["category_risk"] = apply_category_risk(sparkov_test, "category", category_risk_lookup)

# 3. Same six steps as Cell 2, just with category_risk added to the feature list
features = ["amt", "city_pop", "category_risk"]
target = "is_fraud"

X_train = sparkov_train[features]
y_train = sparkov_train[target]
X_test = sparkov_test[features]
y_test = sparkov_test[target]

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

from sklearn.metrics import average_precision_score, fbeta_score, matthews_corrcoef
y_prob = model.predict_proba(X_test_scaled)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

print("PR-AUC:", average_precision_score(y_test, y_prob))
print("F2:", fbeta_score(y_test, y_pred, beta=2))
print("MCC:", matthews_corrcoef(y_test, y_pred))

PR-AUC: 0.1393967891206238
F2: 0.0
MCC: -0.0014871395291730385


Adding category risk moved PR-AUC from 0.1366 to 0.1394, a small increase, while F2 and MCC stayed effectively unchanged. This is a smaller gain than the EDA's 11x spread in fraud rate between categories might suggest. The likely explanation: even the riskiest category tops out at about 1.76% fraud rate, against a 0.386% test base rate. That is a real signal, but still a low absolute number — a feature that only ever raises predicted probability to around 1.76% gives a linear model little room to reach high precision at reasonable recall, which is exactly what PR-AUC measures. A large relative spread between categories and a large PR-AUC contribution are not the same thing, and this result is a clear example of that difference.

In [9]:
# isolate category_risk to see how much signal it carries by itself
features_solo = ["category_risk"]

X_train_solo = sparkov_train[features_solo]
X_test_solo = sparkov_test[features_solo]

scaler_solo = StandardScaler()
X_train_solo_scaled = scaler_solo.fit_transform(X_train_solo)
X_test_solo_scaled = scaler_solo.transform(X_test_solo)

model_solo = LogisticRegression(max_iter=1000)
model_solo.fit(X_train_solo_scaled, y_train)

y_prob_solo = model_solo.predict_proba(X_test_solo_scaled)[:, 1]
y_pred_solo = (y_prob_solo >= 0.5).astype(int)

print("category_risk alone - PR-AUC:", average_precision_score(y_test, y_prob_solo))
print("category_risk alone - F2:", fbeta_score(y_test, y_pred_solo, beta=2))
print("category_risk alone - MCC:", matthews_corrcoef(y_test, y_pred_solo))

# and the actual per-category values, to see the real spread this feature carries
print(sparkov_train.groupby("category")["category_risk"].first().sort_values(ascending=False))

category_risk alone - PR-AUC: 0.009177127556498067
category_risk alone - F2: 0.0
category_risk alone - MCC: 0.0
category
shopping_net      0.017561
misc_net          0.014458
grocery_pos       0.014098
shopping_pos      0.007225
gas_transport     0.004694
misc_pos          0.003139
grocery_net       0.002948
travel            0.002864
entertainment     0.002478
personal_care     0.002424
kids_pets         0.002114
food_dining       0.001651
home              0.001608
health_fitness    0.001549
Name: category_risk, dtype: float64


Used on its own, category_risk scores a PR-AUC of 0.0092, only about 2.4 times the test base rate (0.386%), and far below the 0.1366 that amt and city_pop alone achieved. This confirms the result above was not caused by the feature being suppressed when combined with others — category is genuinely a weak ranking signal by itself. The category values are correct and match the EDA's spread (1.76% for shopping_net down to 0.15% for health_fitness), so this is a gap between a statistically significant spread in aggregate fraud rate and strong individual predictive power, not a data problem. An 11x relative difference between categories still means every category stays under 2% fraud rate, which is not enough on its own for the model to reach high precision at meaningful recall.

F2 and MCC both came back at exactly 0.0 here, meaning the model predicted zero fraud cases at all using category_risk alone. With this level of class imbalance and no class weighting or threshold tuning, a plain 0.5 threshold produces zero or near-zero predicted positives almost regardless of which features are added, so F2 and MCC are expected to stay near zero through the rest of the feature-engineering stage. PR-AUC is therefore the metric that shows whether each feature group is actually helping during this stage; a flat F2 or MCC should not be read as a feature having no effect.

## Hour of Day

In [10]:
# hour-of-day - test raw hour vs an is_night flag, same six-step shape as before

# 1. Extract the raw hour from the timestamp
sparkov_train["hour"] = sparkov_train["trans_date_trans_time"].dt.hour
sparkov_test["hour"] = sparkov_test["trans_date_trans_time"].dt.hour

# 2. Build an is_night flag - EDA found the fraud-rate spike is specifically 10pm-3am
night_hours = [22, 23, 0, 1, 2, 3]
sparkov_train["is_night"] = sparkov_train["hour"].isin(night_hours).astype(int)
sparkov_test["is_night"] = sparkov_test["hour"].isin(night_hours).astype(int)

target = "is_fraud"
y_train = sparkov_train[target]
y_test = sparkov_test[target]

# 3. Test A - raw hour (0-23) as a plain number
features_raw_hour = ["amt", "city_pop", "category_risk", "hour"]
X_train = sparkov_train[features_raw_hour]
X_test = sparkov_test[features_raw_hour]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

y_prob = model.predict_proba(X_test_scaled)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

print("Raw hour   - PR-AUC:", average_precision_score(y_test, y_prob))
print("Raw hour   - F2:", fbeta_score(y_test, y_pred, beta=2))
print("Raw hour   - MCC:", matthews_corrcoef(y_test, y_pred))

# 4. Test B - is_night flag instead of raw hour
features_is_night = ["amt", "city_pop", "category_risk", "is_night"]
X_train = sparkov_train[features_is_night]
X_test = sparkov_test[features_is_night]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

y_prob = model.predict_proba(X_test_scaled)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

print("is_night   - PR-AUC:", average_precision_score(y_test, y_prob))
print("is_night   - F2:", fbeta_score(y_test, y_pred, beta=2))
print("is_night   - MCC:", matthews_corrcoef(y_test, y_pred))

Raw hour   - PR-AUC: 0.12995423457338073
Raw hour   - F2: 0.0
Raw hour   - MCC: -0.0014587044725662018
is_night   - PR-AUC: 0.22296414849693877
is_night   - F2: 0.0
is_night   - MCC: -0.001402597165957869


This result is clear. Raw hour actually made performance slightly worse, with PR-AUC dropping from 0.1394 (category_risk only) to 0.1300 once raw hour was added. A linear model treats hour 23 and hour 1 as numerically distant, even though both fall inside the same overnight risk window, so it wastes its capacity trying to fit a straight-line trend to a pattern that is not linear. The is_night flag, by contrast, raised PR-AUC to 0.2230, up from 0.1394 and a gain of roughly 60%, because it gives the model exactly the shape the EDA found: overnight transactions are risky, daytime transactions are not, with no gradient in between to misrepresent. F2 and MCC stay near zero in both cases, consistent with the pattern already established: those two metrics will not move until imbalance handling changes how the threshold behaves.

This result demonstrates that how a feature is encoded can matter as much as whether the underlying signal exists — the same information, encoded two different ways, went from mildly harmful to the largest single gain so far.

is_night replaces raw hour as the feature carried forward from this point, since raw hour was shown to reduce performance rather than improve it.

## Amount

In [11]:
# look at the amount distribution before deciding how to engineer it
sparkov_train.groupby("is_fraud")["amt"].describe()

,count,mean,std,min,25%,50%,75%,max
is_fraud,,,,,,,,
0,1289169.0,67.667110,154.007971,1.00,9.6100,47.280,82.540,28948.90
1,7506.0,531.320092,390.560070,1.06,245.6625,396.505,900.875,1376.04


Two numbers stand out. The 75th percentile for legitimate transactions is \$82.54, while the 25th percentile for fraudulent transactions is \$245.66 — three-quarters of legitimate transactions fall below the value where even the cheapest quarter of fraud transactions begins. This is a strong, clean separation. However, the relationship is not simply 'higher amount means higher risk': legitimate transactions have a long tail reaching \$28,948.90 (a small number of large but genuine purchases), while fraud is capped at \$1,376.04 and never exceeds that in this data. The pattern the EDA described as 'two distinct bands' is really a range: fraud amounts cluster roughly between \$100 and \$1,400, while legitimate amounts tend to be either well below or, less often, well above that range.

Two candidate encodings were tested against this pattern: a log transform to reduce the influence of the long legitimate tail, and a binary flag marking whether an amount fell inside the \$100-\$1,400 range where fraud concentrates.

In [12]:
# test amount encodings against the is_night-based feature set

import numpy as np

target = "is_fraud"
y_train = sparkov_train[target]
y_test = sparkov_test[target]

# candidate 1: log-transform, tames the long legit tail
sparkov_train["amt_log"] = np.log1p(sparkov_train["amt"])
sparkov_test["amt_log"] = np.log1p(sparkov_test["amt"])

# candidate 2: flag for the $100-$1400 range fraud actually clusters in
sparkov_train["amt_in_fraud_range"] = sparkov_train["amt"].between(100, 1400).astype(int)
sparkov_test["amt_in_fraud_range"] = sparkov_test["amt"].between(100, 1400).astype(int)

# same six steps as every cell so far - scale, fit, score - just reusable
# since we're comparing three feature lists in this one cell
def scale_fit_score(features):
    X_train = sparkov_train[features]
    X_test = sparkov_test[features]
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train_scaled, y_train)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    return {
        "PR-AUC": average_precision_score(y_test, y_prob),
        "F2": fbeta_score(y_test, y_pred, beta=2),
        "MCC": matthews_corrcoef(y_test, y_pred),
    }

print("log1p(amt)        ->", scale_fit_score(["amt_log", "city_pop", "category_risk", "is_night"]))
print("amt_in_fraud_range ->", scale_fit_score(["amt_in_fraud_range", "city_pop", "category_risk", "is_night"]))

log1p(amt)        -> {'PR-AUC': np.float64(0.3060769389608977), 'F2': 0.12955960854092527, 'MCC': np.float64(0.24672900068727713)}
amt_in_fraud_range -> {'PR-AUC': np.float64(0.0990439204621514), 'F2': 0.0, 'MCC': 0.0}


The log-transformed amount produced a clear, large improvement: PR-AUC rose from 0.2230 to 0.3061, and F2 and MCC moved off zero for the first time, to 0.1296 and 0.2467. Every earlier feature group had only changed how the model ranked transactions, measured by PR-AUC. F2 and MCC had stayed near zero because the model's predicted probabilities never crossed the 0.5 threshold. Log-transformed amount is the first feature strong enough to push some genuine predictions past that threshold. This means the model is now correctly identifying some real fraud cases, even with the plain default cutoff and no resampling or class weighting. This is the point where amount, once properly scaled, becomes a primary driver of the model's performance rather than a marginal contributor.

The banded flag produced the opposite result: PR-AUC fell to 0.0990, below even the plain baseline, and F2 and MCC returned to exactly zero. This connects to the amount distribution seen earlier. Amount's relationship to fraud is not a step function, unlike hour's. The day-versus-night pattern was genuinely binary, which is why a flag worked well there. Amount's relationship is graded instead: a \$1,200 transaction is more suspicious than a \$150 one, even though both may fall inside the same \$100-\$1,400 range. Reducing that range to a single flag throws away this graded information. It also groups in many ordinary legitimate transactions that happen to fall in the same dollar range, adding noise instead of signal. This is a clear example of why the shape of a feature's relationship to the target, step-like versus continuous, should guide the choice of encoding, rather than simply whether a difference exists.

The feature set carried forward from this point is amt_log, city_pop, category_risk, and is_night. This combination gives PR-AUC 0.3061, F2 0.1296, and MCC 0.2467. The banded flag is not used further.

## Implied Velocity

In [13]:
# implied velocity - compute across the full timeline so test's first
# transaction per cardholder still has a real "previous transaction" from train
import pandas as pd

from src.utils import add_prev_txn_features

# 1. Tag rows so we can split back apart afterward, then stick train+test together
sparkov_train["_split"] = "train"
sparkov_test["_split"] = "test"
combined = pd.concat([sparkov_train, sparkov_test], ignore_index=True)

# 2. Compute velocity across the full timeline - only looks backward per cardholder,
#    so this doesn't leak any test-set fraud label into anything
combined = add_prev_txn_features(
    combined,
    id_column="cc_num",
    time_column="trans_date_trans_time",
    lat_column="merch_lat",
    lon_column="merch_long",
)

# 3. Split back into train/test using the tag from step 1
sparkov_train = combined[combined["_split"] == "train"].drop(columns=["_split"])
sparkov_test = combined[combined["_split"] == "test"].drop(columns=["_split"])

# 4. Look at the result before deciding how to scale it
print(sparkov_train[["dist_from_prev_txn_km", "time_gap_hours", "implied_velocity_kmh"]].describe())
print("Missing in train:", sparkov_train["implied_velocity_kmh"].isna().sum())
print("Missing in test:", sparkov_test["implied_velocity_kmh"].isna().sum())

       dist_from_prev_txn_km  time_gap_hours  implied_velocity_kmh
count           1.295692e+06    1.295692e+06          1.295692e+06
mean            1.035944e+02    9.023615e+00          1.741713e+02
std             5.039868e+01    1.316694e+01          2.757404e+03
min             8.069198e-02    0.000000e+00          3.682374e-03
25%             6.438102e+01    1.670278e+00          7.599740e+00
50%             1.007818e+02    4.600000e+00          2.001599e+01
75%             1.397061e+02    1.117306e+01          5.876024e+01
max             2.859922e+02    3.726308e+02          7.362208e+05
Missing in train: 983
Missing in test: 16


The missing-value counts confirm the timeline fix worked correctly. The 983 missing values in the training set match exactly the EDA's count of unique cardholders — these are each cardholder's genuine first-ever transaction, where no previous transaction can exist, so a missing value there is correct rather than an error. The 16 missing values in the test set are almost certainly new cardholders who do not appear in the training set at all, so no real previous transaction exists for them even with the combined-timeline approach. Both are the expected, unavoidable form of 'no prior history exists.' The fix itself is confirmed by the low test count: had the feature been computed on the split test set alone, with no continuity from training data, roughly 983 test rows would have been missing instead of 16.

The distribution matches what the EDA had already flagged. Distance and time-gap values look realistic (a maximum of 286 km, and a maximum gap of about 15.5 days between transactions). Implied velocity, however, has a mean of 174 km/h against a median of only 20 km/h, with a maximum of 736,220 km/h, a physically impossible value that confirms this is the merchant-registration-location artifact noted in the EDA, not real travel. Feeding this raw into a scaler would let a small number of extreme rows dominate the scaling and obscure the genuine signal, the same problem solved earlier for amount using log1p, though more extreme here (values span about 8 orders of magnitude, compared to roughly 4 for amount).

In [14]:
# implied velocity - fill debut-transaction NaNs, then test two ways
# of taming the extreme outliers (up to 736,000+ km/h, a known merchant-location artifact)

target = "is_fraud"
y_train = sparkov_train[target]
y_test = sparkov_test[target]

# 1. No previous transaction exists yet for these rows - fill with 0 ("no prior info")
for col in ["dist_from_prev_txn_km", "time_gap_hours", "implied_velocity_kmh"]:
    sparkov_train[col] = sparkov_train[col].fillna(0)
    sparkov_test[col] = sparkov_test[col].fillna(0)

# 2. Candidate A: log-transform, same fix that worked for amount
sparkov_train["velocity_log"] = np.log1p(sparkov_train["implied_velocity_kmh"])
sparkov_test["velocity_log"] = np.log1p(sparkov_test["implied_velocity_kmh"])

# 3. Candidate B: clip at the 99th percentile, learned from train only
cap = sparkov_train["implied_velocity_kmh"].quantile(0.99)
sparkov_train["velocity_clipped"] = sparkov_train["implied_velocity_kmh"].clip(upper=cap)
sparkov_test["velocity_clipped"] = sparkov_test["implied_velocity_kmh"].clip(upper=cap)
print("99th percentile cap (from train):", cap)

def scale_fit_score(features):
    X_train = sparkov_train[features]
    X_test = sparkov_test[features]
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train_scaled, y_train)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    return {
        "PR-AUC": average_precision_score(y_test, y_prob),
        "F2": fbeta_score(y_test, y_pred, beta=2),
        "MCC": matthews_corrcoef(y_test, y_pred),
    }

print("velocity_log     ->", scale_fit_score(["amt_log", "city_pop", "category_risk", "is_night", "velocity_log"]))
print("velocity_clipped ->", scale_fit_score(["amt_log", "city_pop", "category_risk", "is_night", "velocity_clipped"]))

99th percentile cap (from train): 1955.7611217790948
velocity_log     -> {'PR-AUC': np.float64(0.34571374713814906), 'F2': 0.20970214629872974, 'MCC': np.float64(0.35087176785522034)}
velocity_clipped -> {'PR-AUC': np.float64(0.3311856602611441), 'F2': 0.14471057884231536, 'MCC': np.float64(0.268150443094085)}


The log-transformed velocity feature performs better across all three metrics: PR-AUC 0.3457 versus 0.3312, F2 0.2097 versus 0.1447, and MCC 0.3509 versus 0.2682 for the clipped version. Both outperform the 0.3061 reference, so velocity adds value under either encoding. The reason mirrors the amount feature: velocity's relationship to fraud is graded and continuous rather than a threshold effect, and the EDA had already found this relationship 'holds across percentiles, not just driven by outliers.' The log transform preserves that graded structure across the full range while still reducing the influence of the extreme merchant-location artifacts, whereas clipping only addresses the very top of the distribution and recovers less of the underlying signal. velocity_log is used as the final feature.

This completes all four feature groups. The final feature set, amt_log, city_pop, category_risk, is_night, and velocity_log, gives PR-AUC 0.3457, F2 0.2097, and MCC 0.3509, compared with the naive baseline of 0.1366, 0.0, and -0.0016.

## Final Uplift Table

In [15]:
# the full uplift table, built cleanly in one place

FEATURE_STEPS = {
    "baseline (raw amt, city_pop)": ["amt", "city_pop"],
    "+ category_risk": ["amt", "city_pop", "category_risk"],
    "+ is_night": ["amt", "city_pop", "category_risk", "is_night"],
    "+ amt_log (replaces raw amt)": ["amt_log", "city_pop", "category_risk", "is_night"],
    "+ velocity_log": ["amt_log", "city_pop", "category_risk", "is_night", "velocity_log"],
}

rows = {}
for name, features in FEATURE_STEPS.items():
    rows[name] = scale_fit_score(features)

uplift_table = pd.DataFrame(rows).T

baseline_scores = uplift_table.iloc[0]
for metric in ["PR-AUC", "F2", "MCC"]:
    uplift_table[f"{metric} delta"] = uplift_table[metric] - baseline_scores[metric]

uplift_table

,PR-AUC,F2,MCC,PR-AUC delta,F2 delta,MCC delta
"baseline (raw amt, city_pop)",0.136581,0.000000,-0.001583,0.000000,0.000000,0.000000
+ category_risk,0.139397,0.000000,-0.001487,0.002815,0.000000,0.000096
+ is_night,0.222964,0.000000,-0.001403,0.086383,0.000000,0.000180
+ amt_log (replaces raw amt),0.306077,0.129560,0.246729,0.169496,0.129560,0.248312
+ velocity_log,0.345714,0.209702,0.350872,0.209132,0.209702,0.352454


The finalized feature set — amt_log, city_pop, category_risk, is_night, and
velocity_log — improved PR-AUC from 0.137 to 0.346, F2 from 0.0 to 0.210, and
MCC from -0.002 to 0.351, compared with the naive baseline. Most of this
improvement came from two decisions: replacing raw hour with a binary
night-time flag, and using a log transform rather than a threshold-based flag
for both amount and implied velocity. Each choice was confirmed by directly
comparing both options rather than assumed in advance; in both cases, the
untested alternative would have performed worse than not adding the feature
at all.

In [16]:
# drop the columns we tested and didn't keep
sparkov_train = sparkov_train.drop(columns=["hour", "amt_in_fraud_range", "velocity_clipped"], errors="ignore")
sparkov_test = sparkov_test.drop(columns=["hour", "amt_in_fraud_range", "velocity_clipped"], errors="ignore")

print(sparkov_train.columns.tolist())

['trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud', 'category_risk', 'is_night', 'amt_log', 'dist_from_prev_txn_km', 'time_gap_hours', 'implied_velocity_kmh', 'velocity_log']
